<a href="https://colab.research.google.com/github/tushigamarjargal/thesis/blob/main/BTC_lstm_medium.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install keras_tuner

In [2]:
## 1. Import Libraries

# %%
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import time

# Data and Preprocessing
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, mean_squared_error, r2_score
# from sklearn.model_selection import train_test_split # Not directly used for initial split logic here

# LSTM / ANN
# Ensure tensorflow is installed: pip install tensorflow
# Ensure keras-tuner is installed: pip install keras-tuner -U
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import callbacks
import keras_tuner as kt

# Visualization
import plotly.graph_objects as go
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

In [3]:
# ## 2. Configuration

# %%
# --- Defined Parameters ---
ticker = "BTC-USD"
start_date = "2017-11-09"
end_date = "2025-01-01"

# >>> New Parameter <<<
forecast_horizon = 30 # Predict 30 days ahead

# Define Train/Test Split Ratio for the *initial* training phase
train_split_ratio = 0.80
# Define Validation Split Ratio *within* the initial training data for tuning
validation_split_ratio_for_tuning = 0.20 # 20% of initial train data for validation

# LSTM Network Parameters
look_back = 60 # Number of previous days to use for prediction

# Keras Tuner Configuration
MAX_TRIALS = 10 # Number of hyperparameter combinations to try
EXECUTIONS_PER_TRIAL = 2 # Number of models to train per trial
TUNER_EPOCHS = 50 # Max epochs during tuning search
TUNER_PATIENCE = 5 # Early stopping patience during tuning

# Final Training Configuration
FINAL_TRAINING_EPOCHS = 100 # Max epochs for the final model training
FINAL_TRAINING_PATIENCE = 10 # Early stopping patience for final training

# Walk-Forward Configuration
RETRAIN_FREQUENCY = 0 # How often to retrain (0 means train once initially)
# If RETRAIN_FREQUENCY > 0, e.g., 30, retrain every 30 steps
RETRAIN_EPOCHS = 5 # Number of epochs for periodic retraining
FINAL_TRAINING_BATCH_SIZE = 32 # Default batch size if not tuned

# Ensure Keras Tuner directory is cleaned if needed
# !rm -rf ./keras_tuner_lstm_wf_t30/

In [4]:
# ## 3. Data Loading and Preparation

# %%
print(f"--- Loading Data for {ticker} ---")
try:
    df_full = yf.download(tickers=[ticker], start=start_date, end=end_date, progress=False)
    if df_full.empty: raise ValueError(f"No data downloaded for {ticker}.")
    if 'Close' not in df_full.columns: raise ValueError(f"'Close' column not found.")
    df_full = df_full[['Close']].copy()
    # Ensure daily frequency and forward fill missing values (e.g., weekends if source omits them)
    df_full = df_full.asfreq('D')
    df_full.ffill(inplace=True)
    df_full.dropna(inplace=True) # Drop any remaining NaNs (e.g., at the start if fill failed)
    if df_full.empty: raise ValueError(f"Data became empty after processing.")
    print(f"Loaded {len(df_full)} data points for {ticker} from {df_full.index.min().strftime('%Y-%m-%d')} to {df_full.index.max().strftime('%Y-%m-%d')}.")
except Exception as e:
    # Using ValueError for clearer error type in subsequent checks
    raise ValueError(f"Failed to load data for {ticker}: {e}")


# Split Data into initial train+val and test sets
n_total = len(df_full)
n_train_val = int(train_split_ratio * n_total)
n_test = n_total - n_train_val
# Data for initial training and tuning
train_val_data_df = df_full[:n_train_val]
# Data for walk-forward evaluation (Actuals start later due to horizon)
test_data_df = df_full[n_train_val:] # This holds the actuals during the test period

print(f"\nInitial Train+Validation Data: {n_train_val} points ({train_val_data_df.index.min().strftime('%Y-%m-%d')} to {train_val_data_df.index.max().strftime('%Y-%m-%d')})")
# Note: The test period starts here, but evaluation is on t+30 actuals
print(f"Test Period Start Date: {test_data_df.index.min().strftime('%Y-%m-%d')}")
print(f"Number of Walk-Forward Steps: {n_test}")

--- Loading Data for BTC-USD ---
YF.download() has changed argument auto_adjust default to True
Loaded 2610 data points for BTC-USD from 2017-11-09 to 2024-12-31.

Initial Train+Validation Data: 2088 points (2017-11-09 to 2023-07-28)
Test Period Start Date: 2023-07-29
Number of Walk-Forward Steps: 522


In [5]:
# ## 4. Scaling (Fit on Initial Train Portion Only)

# %%
print("\n--- Scaling Data ---")
# Further split train_val_data into train and validation for fitting the scaler
# We fit the scaler ONLY on the data used for the initial training phase of tuning
n_val_tune = int(validation_split_ratio_for_tuning * n_train_val)
n_train_tune = n_train_val - n_val_tune

# Indices for slicing
train_tune_end_idx = n_train_tune
val_tune_end_idx = n_train_val

train_tune_values_for_scaler = df_full['Close'].values[:train_tune_end_idx].reshape(-1, 1)

scaler = MinMaxScaler(feature_range=(0, 1))
# Fit scaler ONLY on the strict initial training portion
scaler.fit(train_tune_values_for_scaler)
print("Scaler fitted on initial training portion (excluding tuning validation).")

# Transform the entire dataset using the fitted scaler
scaled_data = scaler.transform(df_full['Close'].values.reshape(-1, 1))

# Separate scaled train/validation portions for tuning
scaled_train_tune_data = scaled_data[:train_tune_end_idx]
scaled_val_tune_data = scaled_data[train_tune_end_idx:val_tune_end_idx]


--- Scaling Data ---
Scaler fitted on initial training portion (excluding tuning validation).


In [6]:
# ## 5. Sequence Generation Function (Modified for t+h Horizon)

# %%
def create_sequences(data, look_back, forecast_horizon):
    """
    Creates sequences for time series forecasting.

    Args:
        data (np.array): Scaled time series data (n_samples, 1).
        look_back (int): Number of past observations as input features.
        forecast_horizon (int): Number of steps ahead to forecast.

    Returns:
        tuple: (np.array of input sequences (X), np.array of target values (Y))
    """
    X, Y = [], []
    if data.ndim == 1: data = data.reshape(-1, 1)

    # Loop needs to stop early enough to have a target value 'forecast_horizon' steps ahead
    # Max starting index 'i' such that 'i + forecast_horizon - 1' is a valid index
    max_start_index = len(data) - forecast_horizon
    for i in range(look_back, max_start_index + 1):
        X.append(data[i - look_back:i, 0])        # Input sequence ends at index i-1
        Y.append(data[i + forecast_horizon - 1, 0]) # Target is at index i + forecast_horizon - 1
    return np.array(X), np.array(Y)

In [7]:
# ## 6. Prepare Data for Tuning

# %%
print(f"\n--- Preparing Sequences for Tuning (t+{forecast_horizon}) ---")
# Create sequences for the tuning training set
# Use scaled data corresponding to the initial training part
x_train_tune, y_train_tune = create_sequences(scaled_train_tune_data, look_back, forecast_horizon)
x_train_tune = np.reshape(x_train_tune, (x_train_tune.shape[0], x_train_tune.shape[1], 1))

# Create sequences for the tuning validation set
# Need data overlap for lookback, and data extending to the forecast horizon
val_tune_data_start_idx = train_tune_end_idx - look_back
val_tune_data_end_idx = val_tune_end_idx + forecast_horizon - 1 # Needs to include target values

# Check if end index exceeds data length
if val_tune_data_end_idx > n_train_val:
     print(f"Warning: Validation sequence generation requires data beyond initial train+val split end ({n_train_val}).")
     print(f"         Required end index: {val_tune_data_end_idx}. Adjusting.")
     # Adjusting end index, might reduce validation set size implicitly if needed
     val_tune_data_end_idx = min(val_tune_data_end_idx, len(scaled_data))

# Ensure start index is valid
val_tune_data_start_idx = max(0, val_tune_data_start_idx)

val_tune_data_for_seq = scaled_data[val_tune_data_start_idx : val_tune_data_end_idx]

# Generate validation sequences
x_val_tune, y_val_tune = create_sequences(val_tune_data_for_seq, look_back, forecast_horizon)

# We only need validation sequences that correspond to the original validation time frame
# The 'i' in create_sequences corresponds to the *end* of the input sequence.
# Original validation data indices: train_tune_end_idx to val_tune_end_idx - 1
# Corresponding 'i' values: train_tune_end_idx to val_tune_end_idx -1
# So, the first validation sequence corresponds to i = train_tune_end_idx
# Find the index in x_val_tune/y_val_tune corresponding to this first actual validation point
first_val_i_in_full_data = train_tune_end_idx
first_val_i_in_seq_data = first_val_i_in_full_data - val_tune_data_start_idx
# The sequence index is `first_val_i_in_seq_data - look_back`
val_start_idx_in_seq = max(0, first_val_i_in_seq_data - look_back)


# Ensure we don't exceed the number of generated validation sequences
num_generated_val_seq = x_val_tune.shape[0]
val_start_idx_in_seq = min(val_start_idx_in_seq, num_generated_val_seq)


print(f"Adjusted val_start_idx_in_seq: {val_start_idx_in_seq}")
if val_start_idx_in_seq >= num_generated_val_seq:
    print("Warning: No valid validation sequences could be generated with the current setup. Check splits/horizon.")
    x_val_tune = np.array([])
    y_val_tune = np.array([])
else:
    x_val_tune = x_val_tune[val_start_idx_in_seq:]
    y_val_tune = y_val_tune[val_start_idx_in_seq:]
    x_val_tune = np.reshape(x_val_tune, (x_val_tune.shape[0], x_val_tune.shape[1], 1))


print('x_train_tune shape:', x_train_tune.shape)
print('y_train_tune shape:', y_train_tune.shape)
print('x_val_tune shape:', x_val_tune.shape)
print('y_val_tune shape:', y_val_tune.shape)

if x_val_tune.shape[0] == 0 or y_val_tune.shape[0] == 0:
    raise ValueError("Validation set for tuning is empty. Check data splits, look_back, and forecast_horizon.")



--- Preparing Sequences for Tuning (t+30) ---
         Required end index: 2117. Adjusting.
Adjusted val_start_idx_in_seq: 0
x_train_tune shape: (1582, 60, 1)
y_train_tune shape: (1582,)
x_val_tune shape: (417, 60, 1)
y_val_tune shape: (417,)


In [8]:
# ## 7. LSTM Model Building Function for Keras Tuner

# %%
def build_model(hp):
    """Builds LSTM model with tunable hyperparameters."""
    model = Sequential()
    model.add(LSTM(units=hp.Int('units_1', min_value=32, max_value=128, step=32),
                   return_sequences=True, # True for stacking LSTM layers
                   input_shape=(look_back, 1))) # Input shape defined by look_back
    model.add(Dropout(rate=hp.Float('dropout_1', min_value=0.0, max_value=0.3, step=0.1)))

    # Second LSTM layer
    model.add(LSTM(units=hp.Int('units_2', min_value=32, max_value=128, step=32),
                   return_sequences=False)) # False before Dense layer
    model.add(Dropout(rate=hp.Float('dropout_2', min_value=0.0, max_value=0.3, step=0.1)))

    # Dense layer for further processing
    model.add(Dense(units=hp.Int('dense_units', min_value=16, max_value=64, step=16), activation='relu')) # Added activation

    # Output layer: Predicts a single value (price at t+forecast_horizon)
    model.add(Dense(1))

    # Compile the model
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    # Batch size is handled during fit, not model build typically, but included here as per original code
    hp_batch_size = hp.Choice('batch_size', values=[16, 32, 64]) # This HP influences training speed/stability

    model.compile(optimizer=Adam(learning_rate=hp_learning_rate),
                  loss='mean_squared_error') # Regression problem
    return model

In [9]:
# ## 8. Hyperparameter Tuning

# %%
print(f"\n--- Starting Hyperparameter Search with Keras Tuner (t+{forecast_horizon}) ---")
tuner = kt.RandomSearch(
    build_model,
    objective='val_loss', # Aim to minimize validation loss
    max_trials=MAX_TRIALS,
    executions_per_trial=EXECUTIONS_PER_TRIAL,
    directory='keras_tuner_lstm_wf_t30', # Separate directory for t+30 tuning
    project_name=f'{ticker}_lstm_wf_tuning_t{forecast_horizon}'
)

tuner_early_stopping = callbacks.EarlyStopping(monitor='val_loss', patience=TUNER_PATIENCE, verbose=1)

# Use the prepared tuning sequences
tuner.search(x_train_tune, y_train_tune,
             epochs=TUNER_EPOCHS,
             validation_data=(x_val_tune, y_val_tune),
             callbacks=[tuner_early_stopping],
             verbose=1)

# Get the optimal hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

# Retrieve the best batch size from hyperparameters
tuned_batch_size = best_hps.get('batch_size')

print(f"""
--- Hyperparameter Search Complete (t+{forecast_horizon}) ---
Best Hyperparameters Found:
- LSTM Layer 1 Units: {best_hps.get('units_1')}
- Dropout 1 Rate: {best_hps.get('dropout_1'):.2f}
- LSTM Layer 2 Units: {best_hps.get('units_2')}
- Dropout 2 Rate: {best_hps.get('dropout_2'):.2f}
- Dense Layer Units: {best_hps.get('dense_units')}
- Learning Rate: {best_hps.get('learning_rate')}
- Batch Size: {tuned_batch_size}
""")

Trial 10 Complete [00h 01m 40s]
val_loss: 0.002885125926695764

Best val_loss So Far: 0.0028264992870390415
Total elapsed time: 00h 17m 26s

--- Hyperparameter Search Complete (t+30) ---
Best Hyperparameters Found:
- LSTM Layer 1 Units: 32
- Dropout 1 Rate: 0.10
- LSTM Layer 2 Units: 128
- Dropout 2 Rate: 0.20
- Dense Layer Units: 16
- Learning Rate: 0.0001
- Batch Size: 16



In [10]:
# ## 9. Train Final Initial Model with Best Hyperparameters

# %%
print(f"\n--- Training Final Initial LSTM Model (t+{forecast_horizon}) ---")
start_time_initial_train = time.time()

# Prepare sequences using the *entire* initial train+validation data
scaled_train_val_data = scaled_data[:n_train_val + forecast_horizon -1] # Need data up to the last target

x_train_val_final, y_train_val_final = create_sequences(scaled_train_val_data, look_back, forecast_horizon)

# Ensure we only train on sequences where the *input* is within the original train_val period
# Max 'i' where input sequence ends within train_val period: n_train_val
# Corresponding sequence index = n_train_val - look_back
num_final_train_sequences = n_train_val - look_back
x_train_val_final = x_train_val_final[:num_final_train_sequences]
y_train_val_final = y_train_val_final[:num_final_train_sequences]

x_train_val_final = np.reshape(x_train_val_final, (x_train_val_final.shape[0], x_train_val_final.shape[1], 1))

print(f"Final training sequences shape: X={x_train_val_final.shape}, Y={y_train_val_final.shape}")

# Build the final model with the best hyperparameters directly from tuner
final_lstm_model = tuner.hypermodel.build(best_hps)

# Define early stopping for the final training phase (monitor training loss)
final_early_stopping = callbacks.EarlyStopping(
    monitor='loss',
    patience=FINAL_TRAINING_PATIENCE,
    restore_best_weights=True, # Restore model weights from the epoch with the best monitor value
    verbose=1
)

print(f"Training final initial LSTM on {x_train_val_final.shape[0]} sequences for up to {FINAL_TRAINING_EPOCHS} epochs...")
history_final = final_lstm_model.fit(
    x_train_val_final, y_train_val_final,
    epochs=FINAL_TRAINING_EPOCHS,
    batch_size=tuned_batch_size, # Use tuned batch size
    callbacks=[final_early_stopping],
    verbose=1 # Show training progress
)

end_time_initial_train = time.time()
print(f"Final Initial LSTM Training Complete in {end_time_initial_train - start_time_initial_train:.2f} seconds.")
final_lstm_model.summary()


--- Training Final Initial LSTM Model (t+30) ---
Final training sequences shape: X=(2028, 60, 1), Y=(2028,)
Training final initial LSTM on 2028 sequences for up to 100 epochs...
Epoch 1/100
127/127 ━━━━━━━━━━━━━━━━━━━━ 15s 70ms/step - loss: 0.0657
Epoch 2/100
127/127 ━━━━━━━━━━━━━━━━━━━━ 10s 81ms/step - loss: 0.0119
Epoch 3/100
127/127 ━━━━━━━━━━━━━━━━━━━━ 11s 83ms/step - loss: 0.0100
Epoch 4/100
127/127 ━━━━━━━━━━━━━━━━━━━━ 10s 79ms/step - loss: 0.0098
Epoch 5/100
127/127 ━━━━━━━━━━━━━━━━━━━━ 11s 87ms/step - loss: 0.0087
Epoch 6/100
127/127 ━━━━━━━━━━━━━━━━━━━━ 10s 77ms/step - loss: 0.0082
Epoch 7/100
127/127 ━━━━━━━━━━━━━━━━━━━━ 11s 82ms/step - loss: 0.0081
Epoch 8/100
127/127 ━━━━━━━━━━━━━━━━━━━━ 10s 81ms/step - loss: 0.0073
Epoch 9/100
127/127 ━━━━━━━━━━━━━━━━━━━━ 10s 80ms/step - loss: 0.0079
Epoch 10/100
127/127 ━━━━━━━━━━━━━━━━━━━━ 13s 105ms/step - loss: 0.0076
Epoch 11/100
127/127 ━━━━━━━━━━━━━━━━━━━━ 14s 109ms/step - loss: 0.0074
Epoch 12/100
127/127 ━━━━━━━━━━━━━━━━━━━━ 11s 8

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 60, 32)         │         4,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 60, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 128)            │        82,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │         2,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 266,597 (1.02 MB)

 Trainable params: 88,865 (347.13 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 177,732 (694.27 KB)

In [11]:
# ## 10. Walk-Forward Validation (Rolling Forecast) Loop

# %%
print(f"\n--- Starting LSTM Walk-Forward Validation for {n_test} steps (t+{forecast_horizon}) ---")
start_time_walk_forward = time.time()

lstm_walk_forward_predictions = []
# Initialize history with all scaled data up to the start of the test period
# The history contains the actual scaled values observed so far
history_scaled = scaled_data[:n_train_val].flatten().tolist()

for i in range(n_test):
    # 1. Define the exact index for the current step in the full dataset
    # This index marks the end of the data *currently available* for making the forecast
    current_full_index = n_train_val + i

    # 2. Prepare the input sequence using the *most recent* 'look_back' points from history
    if len(history_scaled) < look_back:
         raise IndexError(f"History length ({len(history_scaled)}) too short for look_back ({look_back}) at step {i}.")
    # Input sequence uses data from history_scaled[-(look_back):]
    input_sequence = np.array(history_scaled[-look_back:]).reshape((1, look_back, 1))

    # 3. Predict t+forecast_horizon ahead (scaled) using the *final_lstm_model*
    # The model directly predicts the value 'forecast_horizon' steps ahead
    pred_scaled = final_lstm_model.predict(input_sequence, verbose=0)[0, 0]

    # 4. Inverse transform the prediction to get the actual price forecast
    pred_unscaled = scaler.inverse_transform([[pred_scaled]])[0, 0]
    lstm_walk_forward_predictions.append(pred_unscaled)

    # --- Update History with ACTUAL value that just became available ---
    # 5. Get the actual scaled value for the current time step `i` in the test set
    # This corresponds to index `current_full_index` in the full `scaled_data` array
    if current_full_index >= len(scaled_data):
        print(f"Warning: Reached end of available data at step {i}. Cannot append actual value.")
        # Optionally break or handle prediction storage if needed
        break
    actual_scaled_value_i = scaled_data[current_full_index, 0]

    # 6. Append the *actual* scaled value to the history for the next prediction step
    history_scaled.append(actual_scaled_value_i)

    # --- Periodic Retraining (Optional) ---
    if RETRAIN_FREQUENCY > 0 and (i + 1) % RETRAIN_FREQUENCY == 0 and (i + 1) < n_test :
        print(f"\n--- Retraining LSTM at step {i+1}/{n_test} (t+{forecast_horizon}) ---")
        retrain_start_time = time.time()

        # Use all available history up to this point for retraining
        current_history_array = np.array(history_scaled).reshape(-1, 1)

        # Generate sequences using the updated history
        x_retrain, y_retrain = create_sequences(current_history_array, look_back, forecast_horizon)

        # Ensure retraining data is valid
        if x_retrain.shape[0] > 0 :
             x_retrain = np.reshape(x_retrain, (x_retrain.shape[0], x_retrain.shape[1], 1))
             print(f"Retraining on {len(x_retrain)} sequences...")
             # Retrain the existing model for a few epochs
             final_lstm_model.fit(x_retrain, y_retrain,
                                  epochs=RETRAIN_EPOCHS,
                                  batch_size=tuned_batch_size, # Use tuned batch size
                                  verbose=0) # Keep retraining less verbose
             retrain_end_time = time.time()
             print(f"Retraining complete in {retrain_end_time - retrain_start_time:.2f} seconds.")
        else:
             print("Not enough data in history to generate sequences for retraining. Skipping.")
        # --- End Retraining ---

    # Log progress periodically
    elif (i + 1) % 100 == 0:
        print(f"LSTM Walk-Forward Step {i+1}/{n_test} complete.")


end_time_walk_forward = time.time()
total_walk_forward_time = end_time_walk_forward - start_time_walk_forward
print(f"\nLSTM Walk-Forward finished in {total_walk_forward_time:.2f} seconds.")

# Ensure predictions list is numpy array
lstm_walk_forward_predictions = np.array(lstm_walk_forward_predictions)


--- Starting LSTM Walk-Forward Validation for 522 steps (t+30) ---
LSTM Walk-Forward Step 100/522 complete.
LSTM Walk-Forward Step 200/522 complete.
LSTM Walk-Forward Step 300/522 complete.
LSTM Walk-Forward Step 400/522 complete.
LSTM Walk-Forward Step 500/522 complete.

LSTM Walk-Forward finished in 96.20 seconds.


In [12]:
# ## 11. Evaluate Walk-Forward Performance

# %%
# Define the evaluation metrics function
def evaluate_forecast(y_true, y_pred, model_name, horizon):
    """Calculates and prints standard evaluation metrics."""
    if len(y_true) == 0 or len(y_pred) == 0:
        print(f"\n--- {model_name} Walk-Forward (t+{horizon}) Evaluation Results ---")
        print("Evaluation skipped: No data points to evaluate.")
        return {'RMSE': np.nan, 'MAE': np.nan, 'MAPE': np.nan, 'R2': np.nan}

    y_true_flat = y_true.flatten()
    y_pred_flat = y_pred.flatten()
    mae = mean_absolute_error(y_true_flat, y_pred_flat)
    mape = mean_absolute_percentage_error(y_true_flat, y_pred_flat)
    rmse = np.sqrt(mean_squared_error(y_true_flat, y_pred_flat))
    try:
        r2 = r2_score(y_true_flat, y_pred_flat)
    except ValueError: # Can happen if y_true is constant
        r2 = np.nan
    print(f"\n--- {model_name} Walk-Forward (t+{horizon}) Evaluation Results ---")
    print(f"RMSE: {rmse:.4f}, MAE: {mae:.4f}, MAPE: {mape:.4%}, R²: {r2:.4f}")
    print(f"Number of evaluation points: {len(y_true_flat)}")
    return {'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'R2': r2}

# --- Correct Alignment for Evaluation ---
# Predictions made during the loop correspond to future dates.
# The prediction made at step `i` (using data up to `n_train_val + i - 1`)
# forecasts the value for date `df_full.index[n_train_val + i + forecast_horizon - 1]`

# Determine the indices of the actual values corresponding to the predictions
start_actual_idx = n_train_val + forecast_horizon - 1
end_actual_idx = n_train_val + n_test + forecast_horizon - 1 # Theoretical end index

# Ensure the end index does not exceed the total data length
max_actual_idx = len(df_full)
end_actual_idx = min(end_actual_idx, max_actual_idx)

# Extract the actual values for comparison
y_test_actual = df_full['Close'].values[start_actual_idx : end_actual_idx]

# Trim predictions if necessary (if the loop generated predictions beyond available actuals)
num_eval_points = len(y_test_actual)
lstm_walk_forward_predictions_eval = lstm_walk_forward_predictions[:num_eval_points]


print(f"\n--- Evaluation Alignment ---")
print(f"Number of predictions made: {len(lstm_walk_forward_predictions)}")
print(f"Actual data start index for eval: {start_actual_idx}")
print(f"Actual data end index for eval (exclusive): {end_actual_idx}")
print(f"Number of actual data points for eval: {len(y_test_actual)}")
print(f"Number of predictions used for eval: {len(lstm_walk_forward_predictions_eval)}")

if len(y_test_actual) != len(lstm_walk_forward_predictions_eval):
     # This check should ideally not fail if logic above is correct, but good failsafe
    raise ValueError(f"Length mismatch after alignment: Actual ({len(y_test_actual)}) vs Predictions ({len(lstm_walk_forward_predictions_eval)})")

# Evaluate against the correctly aligned actual test data
lstm_wf_results = evaluate_forecast(y_test_actual, lstm_walk_forward_predictions_eval, f"LSTM ({ticker})", forecast_horizon)


--- Evaluation Alignment ---
Number of predictions made: 522
Actual data start index for eval: 2117
Actual data end index for eval (exclusive): 2610
Number of actual data points for eval: 493
Number of predictions used for eval: 493

--- LSTM (BTC-USD) Walk-Forward (t+30) Evaluation Results ---
RMSE: 19889.1265, MAE: 15198.7965, MAPE: 22.6266%, R²: -0.0650
Number of evaluation points: 493


In [13]:
# ## 12. Visualize Walk-Forward Results

# %%
print("\n--- Plotting Walk-Forward Forecasts ---")

# Get the correct dates for the x-axis (corresponding to the target forecast dates)
actual_dates = df_full.index[start_actual_idx : end_actual_idx]

if len(actual_dates) != len(lstm_walk_forward_predictions_eval):
     raise ValueError("Date length mismatch for plotting.")

results_df_wf = pd.DataFrame({
    'Actual': y_test_actual.flatten(),
    f'LSTM (t+{forecast_horizon})': lstm_walk_forward_predictions_eval.flatten()
}, index=actual_dates) # Use the target dates as index

fig = go.Figure()
# Plot actual data for the evaluation period
fig.add_trace(go.Scatter(x=results_df_wf.index, y=results_df_wf['Actual'], mode='lines', name='Actual Price (Eval Period)', line=dict(color='black')))
# Plot the t+30 forecasts
fig.add_trace(go.Scatter(x=results_df_wf.index, y=results_df_wf[f'LSTM (t+{forecast_horizon})'], mode='lines', name=f'LSTM Walk-Forward (t+{forecast_horizon})', line=dict(color='orange', dash='dash')))

# Optionally add the full actual data for context
# fig.add_trace(go.Scatter(x=df_full.index, y=df_full['Close'], mode='lines', name='Actual Price (Full)', line=dict(color='grey', width=1), opacity=0.5))


fig.update_layout(
    title=f'LSTM Walk-Forward (t+{forecast_horizon}) Forecast Comparison for {ticker} (Tuned)',
    xaxis_title="Date (Target Date of Forecast)",
    yaxis_title="Price (USD)",
    legend_title="Data/Model",
    template="plotly_white"
)
fig.show()


--- Plotting Walk-Forward Forecasts ---


In [14]:
# ## 13. Walk-Forward Evaluation Period Summary

# %%
print(f"\n--- Walk-Forward Evaluation Summary (t+{forecast_horizon}) ---")
print(f"Initial Train+Validation Data End Date: {train_val_data_df.index.max().strftime('%Y-%m-%d')}")
print(f"Number of Walk-Forward Steps Made: {n_test}")
print(f"Number of Forecasts Evaluated: {num_eval_points}")
if num_eval_points > 0:
    print(f"Evaluation Period (Target Dates): {results_df_wf.index.min().strftime('%Y-%m-%d')} to {results_df_wf.index.max().strftime('%Y-%m-%d')}")
else:
    print("Evaluation Period: N/A (No points evaluated)")
print(f"Forecast Horizon: {forecast_horizon} days")


--- Walk-Forward Evaluation Summary (t+30) ---
Initial Train+Validation Data End Date: 2023-07-28
Number of Walk-Forward Steps Made: 522
Number of Forecasts Evaluated: 493
Evaluation Period (Target Dates): 2023-08-27 to 2024-12-31
Forecast Horizon: 30 days
